# Setup Configuration

This notebook defines all configurable variables for the At-Bat Assistant workflow.
Run this notebook first, then all subsequent notebooks will automatically use these settings.

**To customize for your workspace:** Update the variables below, then run all cells.

Everything is managed in code — the notebook will automatically:
- Create the **MLflow experiment** if it doesn't exist
- Create the **Service Principal** if it doesn't exist
- Create the **secret scope** and generate/store **OAuth credentials** if missing
- **Grant permissions** to the SP on your catalog and schema

References:
- [Secrets documentation](https://docs.databricks.com/aws/en/security/secrets/example-secret-workflow)
- [Service Principals documentation](https://docs.databricks.com/aws/en/admin/users-groups/service-principals)

In [ ]:
%pip install -U -qqqq "mlflow>=3.9" databricks-openai databricks-agents "psycopg[binary]"
dbutils.library.restartPython()

## Required Configuration

Infrastructure IDs are loaded from your Databricks **secret scope** (default: `atbat-assistant-secrets`).
They are never stored in code. Before running this notebook for the first time, populate the secrets:

```python
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
scope = "atbat-assistant-secrets"

w.secrets.put_secret(scope=scope, key="warehouse-id",          string_value="<your SQL warehouse ID>")
w.secrets.put_secret(scope=scope, key="genie-space-id",        string_value="<your Genie space ID>")
w.secrets.put_secret(scope=scope, key="lakebase-name",         string_value="<your Lakebase instance name>")
w.secrets.put_secret(scope=scope, key="lakebase-instance-id",  string_value="<your Lakebase instance UUID>")
w.secrets.put_secret(scope=scope, key="vector-search-endpoint", string_value="<your Vector Search endpoint name>")
```

| Secret key | Env var | Description |
|---|---|---|
| `warehouse-id` | `WAREHOUSE_ID` | SQL Warehouse ID for Genie queries |
| `genie-space-id` | `GENIE_SPACE_ID` | Genie Space ID |
| `lakebase-name` | `LAKEBASE_NAME` | Lakebase instance name (conversation memory) |
| `lakebase-instance-id` | `LAKEBASE_INSTANCE_ID` | Lakebase instance UUID |
| `vector-search-endpoint` | `VECTOR_SEARCH_ENDPOINT` | Vector Search endpoint name |

You can alternatively set these as **cluster environment variables** or **notebook-scoped `os.environ`** assignments; the notebook checks env vars first and only falls back to the secret scope.

The following are **optional overrides** (sensible defaults are derived automatically):

| Env var | Default |
|---|---|
| `CATALOG` | `"users"` |
| `SCHEMA` | Derived from your Databricks username (e.g., `jane_doe`) |
| `MLFLOW_EXPERIMENT_LOCATION` | `/Users/<your-email>/at-bat-assistant-update/atbat-assistant-memalign` |
| `ASSIGNED_USERS` | Your Databricks email (comma-separated for multiple) |
| `DATABRICKS_HOST` | Derived from `WorkspaceClient().config.host` |

In [ ]:
import json
import os
from pathlib import Path
from databricks.sdk import WorkspaceClient
import mlflow
w = WorkspaceClient()

# ============================================================================
# CREATE SECRET SCOPE (if it doesn't exist)
# Must happen BEFORE we try to read secrets from it.
# ============================================================================
_SECRET_SCOPE = os.getenv("DATABRICKS_SECRET_SCOPE_NAME", "atbat-assistant-secrets")

try:
    w.secrets.create_scope(scope=_SECRET_SCOPE)
    print(f"Created secret scope '{_SECRET_SCOPE}'")
except Exception as _e:
    if "RESOURCE_ALREADY_EXISTS" in str(_e) or "already exists" in str(_e).lower():
        print(f"Secret scope '{_SECRET_SCOPE}' already exists")
    else:
        raise _e

# ============================================================================
# LOAD INFRASTRUCTURE SECRETS
# On first run, secrets may not exist yet. We use placeholder values so the
# notebook can complete setup (SP, experiment, etc.) and you can populate
# the real values later via:
#   w.secrets.put_secret(scope=scope, key="warehouse-id", string_value="<id>")
# ============================================================================
_INFRA_SECRETS = {
    "WAREHOUSE_ID": "warehouse-id",
    "GENIE_SPACE_ID": "genie-space-id",
    "LAKEBASE_NAME": "lakebase-name",
    "LAKEBASE_INSTANCE_ID": "lakebase-instance-id",
    "VECTOR_SEARCH_ENDPOINT": "vector-search-endpoint",
}

_missing_secrets = []
for _env_var, _secret_key in _INFRA_SECRETS.items():
    if not os.getenv(_env_var):
        try:
            os.environ[_env_var] = dbutils.secrets.get(scope=_SECRET_SCOPE, key=_secret_key)
        except Exception as _e:
            _placeholder = f"PLACEHOLDER_{_env_var}"
            os.environ[_env_var] = _placeholder
            _missing_secrets.append((_env_var, _secret_key))

if _missing_secrets:
    print(f"WARNING: {len(_missing_secrets)} secret(s) not yet configured (using placeholders):")
    for _env_var, _secret_key in _missing_secrets:
        print(f"  - {_env_var} (secret key: '{_secret_key}')")
    print(f"\nTo populate later, run:")
    print(f"  w.secrets.put_secret(scope='{_SECRET_SCOPE}', key='<key>', string_value='<value>')")
    print(f"\nNotebooks 01 (data load) and 02 (tooling) do NOT need these secrets.")
    print(f"You must populate them before running notebook 03+ (agent definition).")
else:
    print(f"Infrastructure config loaded from secret scope '{_SECRET_SCOPE}'")

# ============================================================================
# WORKSPACE CONFIGURATION
# ============================================================================
_current_user = w.current_user.me()
_user_email = _current_user.user_name

CATALOG = os.getenv("CATALOG", "users")
SCHEMA = os.getenv("SCHEMA", "at_bat_assistant")
print(f"Current user: {_user_email}")
print(f"Catalog/Schema: {CATALOG}.{SCHEMA}")

# ============================================================================
# DATA COLLECTION CONFIGURATION
# ============================================================================
SEASONS = [2024, 2025]

# ============================================================================
# MLFLOW CONFIGURATION
# ============================================================================
exp_location = os.getenv(
    "MLFLOW_EXPERIMENT_LOCATION",
    f"/Users/{_user_email}/atbat-assistant-memalign",
)

# Use set_experiment which handles create-or-get in one call
experiment = mlflow.set_experiment(experiment_name=exp_location)
EXPERIMENT_ID = experiment.experiment_id
print(f"Using experiment: {exp_location} (ID: {EXPERIMENT_ID})")

mlflow.set_experiment_tags({
    "mlflow.promptRegistryLocation": f"{CATALOG}.{SCHEMA}",
    "purpose": "baseball_analysis",
    "product": "mlflow",
})
print(f"Linked experiment to Prompt Registry location: {CATALOG}.{SCHEMA}")

# ============================================================================
# PROMPT REGISTRY CONFIGURATION
# ============================================================================
PROMPT_NAME = f"{CATALOG}.{SCHEMA}.atbat_assistant_prompt"

# ============================================================================
# LLM ENDPOINT CONFIGURATION
# ============================================================================
LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4-5"
JUDGE_MODEL = "databricks:/databricks-claude-opus-4-5"
REFLECTION_MODEL = "databricks:/databricks-gpt-5-4"

# ============================================================================
# MODEL REGISTRATION CONFIGURATION
# ============================================================================
MODEL_NAME = "atbat_assistant"
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"

# ============================================================================
# GENIE SPACE CONFIGURATION
# ============================================================================
GENIE_SPACE_ID = os.environ["GENIE_SPACE_ID"]
GENIE_SPACE_NAME = "atbat_assistant_genie"
WAREHOUSE_ID = os.environ["WAREHOUSE_ID"]

GENIE_TABLES = [
    "batter_vectors_median",
    "batter_vectors_mean",
    "dim_batter_team_year",
    "dim_batters",
    "dim_pitchers",
    "dim_pitcher_arsenal",
    "dim_pitcher_team_year",
    "dim_players",
    "pitcher_vectors_median",
    "pitcher_vectors_mean",
    "statcast_pitches",
]

# ============================================================================
# LAKEBASE CONFIGURATION
# ============================================================================
LAKEBASE_NAME = os.environ["LAKEBASE_NAME"]
LAKEBASE_INSTANCE_ID = os.environ["LAKEBASE_INSTANCE_ID"]
try:
    _lb_instance = w.database.get_database_instance(name=LAKEBASE_NAME)
    LAKEBASE_HOST = _lb_instance.read_write_dns
    print(f"Lakebase host resolved via SDK: {LAKEBASE_HOST}")
except Exception as _e:
    LAKEBASE_HOST = f"instance-{LAKEBASE_INSTANCE_ID}.database.cloud.databricks.com"
    print(f"Lakebase host (fallback): {LAKEBASE_HOST} (SDK lookup failed: {_e})")

# ============================================================================
# EVALUATION DATASET CONFIGURATION
# ============================================================================
DATASET_NAME = f"{CATALOG}.{SCHEMA}.atbat_assistant_eval_trace_data"
LABEL_SCHEMA_NAME = "baseball_analysis_base"
LABELING_SESSION_NAME = "atbat_assistant_eval_labeling"
ASSIGNED_USERS = os.getenv("ASSIGNED_USERS", _user_email).split(",")

# ============================================================================
# JUDGE/SCORER CONFIGURATION
# ============================================================================
ALIGNED_JUDGE_NAME = "baseball_analysis_base"
EMBEDDING_MODEL = "databricks:/databricks-bge-large-en"

# ============================================================================
# PROMPT OPTIMIZATION CONFIGURATION
# ============================================================================
OPTIMIZATION_DATASET_NAME = f"{CATALOG}.{SCHEMA}.atbat_assistant_optimization_data"

# ============================================================================
# AGENT SKILLS CONFIGURATION
# ============================================================================
SKILLS_VOLUME_NAME = "agent_skills"
SKILLS_VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{SKILLS_VOLUME_NAME}"
GEPA_SKILLS_VOLUME_NAME = "agent_skills_gepa"
GEPA_SKILLS_VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{GEPA_SKILLS_VOLUME_NAME}"

# ============================================================================
# ALIGNMENT TRACKING CONFIGURATION
# ============================================================================
ALIGNMENT_RUNS_TABLE = f"{CATALOG}.{SCHEMA}.atbat_alignment_runs"

# ============================================================================
# UNITY CATALOG TOOLS CONFIGURATION
# ============================================================================
UC_TOOLS_SCHEMA = f"{CATALOG}.{SCHEMA}"
UC_TOOL_NAMES = [
    f"{UC_TOOLS_SCHEMA}.get_batter_pitcher_matchup",
    f"{UC_TOOLS_SCHEMA}.get_pitcher_tendency_by_count",
    f"{UC_TOOLS_SCHEMA}.get_pitcher_tendency_with_runners",
    f"{UC_TOOLS_SCHEMA}.pitcher_embedding_query",
    f"{UC_TOOLS_SCHEMA}.pitcher_embedding_lookup",
    f"{UC_TOOLS_SCHEMA}.pitcher_arsenal_lookup",
    f"{UC_TOOLS_SCHEMA}.batter_embedding_query",
    f"{UC_TOOLS_SCHEMA}.batter_embedding_lookup",
    f"{UC_TOOLS_SCHEMA}.lookup_player_by_name",
    f"{UC_TOOLS_SCHEMA}.batter_embeddings_for_ids",
    f"{UC_TOOLS_SCHEMA}.recommend_batter_matchups_by_team",
    f"{UC_TOOLS_SCHEMA}.get_team_batters",
]

# ============================================================================
# VECTOR SEARCH INDICES
# ============================================================================
VECTOR_SEARCH_ENDPOINT = os.environ["VECTOR_SEARCH_ENDPOINT"]
BATTER_INDEX_NAME = f"{CATALOG}.{SCHEMA}.batter_vectors_mean_index"
PITCHER_INDEX_NAME = f"{CATALOG}.{SCHEMA}.pitcher_vectors_mean_index"

# ============================================================================
# PROMPT REGISTRY AUTHENTICATION CONFIGURATION
# ============================================================================
USE_OAUTH = True
SECRET_SCOPE_NAME = _SECRET_SCOPE
SERVICE_PRINCIPAL_NAME = os.getenv("SERVICE_PRINCIPAL_NAME", "atbat-assistant-sp")
OAUTH_CLIENT_ID_KEY = os.getenv("OAUTH_CLIENT_ID_KEY", "oauth-client-id")
OAUTH_CLIENT_SECRET_KEY = os.getenv("OAUTH_CLIENT_SECRET_KEY", "oauth-client-secret")
PAT_KEY = os.getenv("PAT_KEY", "databricks-pat")
DATABRICKS_HOST = os.getenv("DATABRICKS_HOST", w.config.host.rstrip("/") + "/")

# Find or create service principal
service_principals = list(w.service_principals.list(filter=f'displayName eq "{SERVICE_PRINCIPAL_NAME}"'))
if service_principals:
    service_principal = service_principals[0]
    OAUTH_CLIENT_ID = service_principal.application_id
    print(f"Found existing service principal '{SERVICE_PRINCIPAL_NAME}' (app_id: {OAUTH_CLIENT_ID})")
else:
    print(f"Service principal '{SERVICE_PRINCIPAL_NAME}' not found. Creating...")
    service_principal = w.service_principals.create(display_name=SERVICE_PRINCIPAL_NAME)
    OAUTH_CLIENT_ID = service_principal.application_id
    print(f"Created service principal '{SERVICE_PRINCIPAL_NAME}' (app_id: {OAUTH_CLIENT_ID})")

    import requests
    _host = w.config.host.rstrip("/")
    _token = w.config.authenticate()
    _headers = {"Authorization": f"Bearer {_token}", "Content-Type": "application/json"} if isinstance(_token, str) else {}
    _resp = w.api_client.do(
        "POST",
        f"/api/2.0/accounts/servicePrincipals/{service_principal.id}/credentials/secrets"
    )
    _client_secret_value = _resp.get("secret")
    _secret_id = _resp.get("secret_id", _resp.get("id", "unknown"))
    print(f"Generated OAuth secret (secret_id: {_secret_id})")

    w.secrets.put_secret(scope=SECRET_SCOPE_NAME, key=OAUTH_CLIENT_ID_KEY, string_value=OAUTH_CLIENT_ID)
    w.secrets.put_secret(scope=SECRET_SCOPE_NAME, key=OAUTH_CLIENT_SECRET_KEY, string_value=_client_secret_value)
    print(f"Stored OAuth credentials in scope '{SECRET_SCOPE_NAME}'")
    del _client_secret_value

# ============================================================================
# ASSEMBLE CONFIG DICTIONARY
# ============================================================================
CONFIG = {
    "workspace": {
        "catalog": CATALOG,
        "schema": SCHEMA,
    },
    "data_collection": {
        "seasons": SEASONS,
    },
    "mlflow": {
        "experiment_id": EXPERIMENT_ID,
    },
    "prompt_registry": {
        "prompt_name": PROMPT_NAME,
        "reflection_model": REFLECTION_MODEL,
    },
    "llm": {
        "endpoint_name": LLM_ENDPOINT_NAME,
        "judge_model": JUDGE_MODEL,
    },
    "model": {
        "model_name": MODEL_NAME,
        "uc_model_name": UC_MODEL_NAME,
    },
    "genie": {
        "space_id": GENIE_SPACE_ID,
        "name": GENIE_SPACE_NAME,
        "warehouse_id": WAREHOUSE_ID,
        "tables": GENIE_TABLES,
    },
    "lakebase": {
        "instance_name": LAKEBASE_NAME,
        "instance_id": LAKEBASE_INSTANCE_ID,
        "host": LAKEBASE_HOST,
        "conn_host": LAKEBASE_HOST,
    },
    "evaluation": {
        "dataset_name": DATASET_NAME,
        "label_schema_name": LABEL_SCHEMA_NAME,
        "labeling_session_name": LABELING_SESSION_NAME,
        "assigned_users": ASSIGNED_USERS,
    },
    "judges": {
        "aligned_judge_name": ALIGNED_JUDGE_NAME,
        "embedding_model": EMBEDDING_MODEL,
    },
    "optimization": {
        "optimization_dataset_name": OPTIMIZATION_DATASET_NAME,
    },
    "alignment": {
        "alignment_runs_table": ALIGNMENT_RUNS_TABLE,
    },
    "skills": {
        "volume_name": SKILLS_VOLUME_NAME,
        "volume_path": SKILLS_VOLUME_PATH,
        "gepa_volume_name": GEPA_SKILLS_VOLUME_NAME,
        "gepa_volume_path": GEPA_SKILLS_VOLUME_PATH,
    },
    "tools": {
        "uc_tool_names": UC_TOOL_NAMES,
    },
    "vector_search": {
        "endpoint_name": VECTOR_SEARCH_ENDPOINT,
        "batter_index_name": BATTER_INDEX_NAME,
        "pitcher_index_name": PITCHER_INDEX_NAME,
    },
    "prompt_registry_auth": {
        "use_oauth": USE_OAUTH,
        "secret_scope_name": SECRET_SCOPE_NAME,
        "oauth_client_id_key": OAUTH_CLIENT_ID_KEY,
        "oauth_client_secret_key": OAUTH_CLIENT_SECRET_KEY,
        "oauth_client_id": OAUTH_CLIENT_ID,
        "pat_key": PAT_KEY,
        "databricks_host": DATABRICKS_HOST,
    },
}

print("Configuration details created")
if _missing_secrets:
    print(f"\nREMINDER: {len(_missing_secrets)} infrastructure secret(s) still need to be populated.")

### Step 2: Ensure SP Credentials & Secret Scope

This cell handles the full lifecycle:
1. **Secret scope** — creates it if it doesn't exist
2. **OAuth secret** — generates one for the SP if credentials aren't already stored
3. **Grants** — gives the SP access to the catalog and schema

This is idempotent — safe to re-run. If everything is already configured, it skips.

In [ ]:
# ---- 1. Verify / Create Secret Scope ----
scope_exists = False
try:
    scopes = dbutils.secrets.listScopes()
    scope_exists = any(scope.name == SECRET_SCOPE_NAME for scope in scopes)
except Exception as e:
    print(f"WARNING: Error listing scopes: {e}")

if scope_exists:
    print(f"Secret scope '{SECRET_SCOPE_NAME}' already exists")
else:
    print(f"Secret scope '{SECRET_SCOPE_NAME}' not found. Creating...")
    try:
        w.secrets.create_scope(scope=SECRET_SCOPE_NAME)
        scope_exists = True
        print(f"Created secret scope '{SECRET_SCOPE_NAME}'")
    except Exception as e:
        if "RESOURCE_ALREADY_EXISTS" in str(e) or "already exists" in str(e).lower():
            scope_exists = True
            print(f"Secret scope '{SECRET_SCOPE_NAME}' already exists (race condition)")
        else:
            raise e

# ---- 2. Check if OAuth credentials are stored ----
credentials_stored = False
if scope_exists:
    try:
        secrets_list = dbutils.secrets.list(scope=SECRET_SCOPE_NAME)
        secret_keys = [s.key for s in secrets_list]
        has_client_id = OAUTH_CLIENT_ID_KEY in secret_keys
        has_client_secret = OAUTH_CLIENT_SECRET_KEY in secret_keys
        credentials_stored = has_client_id and has_client_secret
        if credentials_stored:
            print(f"OAuth credentials already stored in scope '{SECRET_SCOPE_NAME}'")
            print(f"  - {OAUTH_CLIENT_ID_KEY}: present")
            print(f"  - {OAUTH_CLIENT_SECRET_KEY}: present")
        else:
            missing = []
            if not has_client_id:
                missing.append(OAUTH_CLIENT_ID_KEY)
            if not has_client_secret:
                missing.append(OAUTH_CLIENT_SECRET_KEY)
            print(f"Missing secrets in scope: {missing}")
    except Exception as e:
        print(f"Could not list secrets: {e}")

# ---- 3. Generate OAuth secret and store if needed ----
if not credentials_stored:
    print(f"\nGenerating OAuth secret for SP '{SERVICE_PRINCIPAL_NAME}'...")

    # Look up the SP
    sp_list = list(w.service_principals.list(filter=f'displayName eq "{SERVICE_PRINCIPAL_NAME}"'))
    if not sp_list:
        raise ValueError(f"Service principal '{SERVICE_PRINCIPAL_NAME}' not found. Run the config cell first.")

    sp = sp_list[0]
    print(f"  SP: {sp.display_name} (id: {sp.id}, app_id: {sp.application_id})")

    # Generate a new OAuth secret via REST API
    try:
        resp = w.api_client.do(
            "POST",
            f"/api/2.0/accounts/servicePrincipals/{sp.id}/credentials/secrets"
        )
        _client_secret_value = resp.get("secret")
        _secret_id = resp.get("secret_id", resp.get("id", "unknown"))
        print(f"  Generated OAuth secret (secret_id: {_secret_id})")
    except Exception as e:
        raise RuntimeError(
            f"Failed to generate OAuth secret: {e}\n"
            "You may need to generate the secret manually via the Databricks UI:\n"
            "  Settings > Identity > Service Principals > Select SP > Generate Secret"
        )

    # Store credentials in the secret scope
    w.secrets.put_secret(scope=SECRET_SCOPE_NAME, key=OAUTH_CLIENT_ID_KEY, string_value=sp.application_id)
    w.secrets.put_secret(scope=SECRET_SCOPE_NAME, key=OAUTH_CLIENT_SECRET_KEY, string_value=_client_secret_value)
    print(f"  Stored credentials in scope '{SECRET_SCOPE_NAME}':")
    print(f"    - {OAUTH_CLIENT_ID_KEY} = {sp.application_id}")
    print(f"    - {OAUTH_CLIENT_SECRET_KEY} = (stored securely)")

    # Clean up
    del _client_secret_value
    credentials_stored = True

# ---- 4. Grant SP access to catalog/schema ----
sp_list = list(w.service_principals.list(filter=f'displayName eq "{SERVICE_PRINCIPAL_NAME}"'))
if sp_list:
    application_id = sp_list[0].application_id
    print(f"\nGranting access to SP '{SERVICE_PRINCIPAL_NAME}' (app_id: {application_id})...")

    spark.sql(f"GRANT USAGE ON CATALOG {CATALOG} TO `{application_id}`")
    spark.sql(f"GRANT USAGE ON SCHEMA {CATALOG}.{SCHEMA} TO `{application_id}`")
    spark.sql(f"GRANT CREATE FUNCTION, EXECUTE, MANAGE ON SCHEMA {CATALOG}.{SCHEMA} TO `{application_id}`")
    print("Grant statements executed successfully")
else:
    print(f"WARNING: Service principal '{SERVICE_PRINCIPAL_NAME}' not found. Cannot grant permissions.")

print("\n--- Summary ---")
print(f"  Secret scope: {'ready' if scope_exists else 'MISSING'}")
print(f"  OAuth credentials: {'stored' if credentials_stored else 'MISSING'}")
print(f"  SP grants: {'applied' if sp_list else 'SKIPPED'}")

### Step 3: Grant SP Access to Lakebase (Provisioned Instance)

This step configures your **service principal** to connect to an **existing shared Lakebase Provisioned instance** for conversation memory (LangGraph checkpointing). Everything is fully automated — no manual UI or SQL Editor steps required, even on shared instances you didn't create.

The setup below runs **3 steps** in order of importance ([docs](https://docs.databricks.com/aws/en/oltp/instances/pg-roles)):

---

#### Step 1: Postgres Roles (who can connect) — **Required**

Every identity (user, SP, group) needs a **Postgres role** before it can connect. Roles are NOT auto-created. Without a role, you get `FATAL: role does not exist`.

| Role | Created via | Why |
|---|---|---|
| **Your user** (`domainexpert@email.com`) | `w.database.create_database_instance_role()` | Needed to connect and run GRANT statements for the SP |
| **Service principal** (`{application_id}`) | `w.database.create_database_instance_role()` | Needed for the agent to connect at runtime |

**Created automatically** via the Databricks SDK — same API the Lakebase App UI calls.

---

#### Step 2: Database Permissions (what can they do once connected) — **Required**

After the SP's Postgres role exists, it only has basic `LOGIN` privilege. We must `GRANT` additional permissions.

| Grant | What it allows |
|---|---|
| `USAGE` + `CREATE` on `public` schema | SP can see the schema and create LangGraph checkpoint tables |
| `ALL PRIVILEGES` on all tables & sequences | SP can read/write checkpoint data |
| `ALTER DEFAULT PRIVILEGES` | Future tables/sequences inherit the same grants |

**Granted automatically** by connecting as your user and executing SQL.

---

#### Step 3: Platform Permissions (who can call the Databricks API) — **Optional / Manual**

Controls who can generate database credentials via `generate_database_credential()` at runtime.

| Permission | What it allows |
|---|---|
| `CAN_MANAGE` | Generate credentials, manage computes, manage settings, create roles for others |
| `CAN_USE` | Generate credentials, view instance (minimum needed) |

The notebook attempts to grant this via the REST API, but **this API does not work reliably for Provisioned instances**. If it fails, **grant access manually**:
1. Open the **Lakebase App** > click your instance
2. Go to **Settings > Permissions**
3. Add the service principal with **CAN_USE** (or CAN_MANAGE)

> Without project-level access, the SP cannot call `generate_database_credential()` and the deployed agent will fail to connect.

---

> **Security note:** The grants are `ALL PRIVILEGES` scoped to the `public` schema only. To restrict further, grant only on the specific checkpoint tables created by LangGraph (`checkpoints`, `checkpoint_blobs`, `checkpoint_writes`).

In [ ]:
import psycopg
import uuid as _uuid
from databricks.sdk.service.database import (
    DatabaseInstanceRole,
    DatabaseInstanceRoleIdentityType,
)

sp_list = list(w.service_principals.list(filter=f'displayName eq "{SERVICE_PRINCIPAL_NAME}"'))
if not sp_list:
    print(f"ERROR: Service principal '{SERVICE_PRINCIPAL_NAME}' not found. Run previous cells first.")
else:
    sp = sp_list[0]
    sp_role = sp.application_id
    current_user = w.current_user.me().user_name
    print(f"SP: '{sp.display_name}' (application_id: {sp_role})")
    print(f"Current user: {current_user}")

    # ---- 1. Create Postgres roles via SDK (no Postgres connection needed) ----
    # Uses w.database.create_database_instance_role() — the same API the Lakebase App UI calls.
    # This works even on shared instances you didn't create, as long as you have CAN_MANAGE.
    # Docs: https://docs.databricks.com/aws/en/oltp/instances/pg-roles
    print(f"\n--- Step 1: Create Postgres roles (SDK API) ---")

    # Check which roles already exist
    existing_roles = {
        r.name for r in w.database.list_database_instance_roles(instance_name=LAKEBASE_NAME)
    }
    print(f"  Existing roles: {len(existing_roles)} found")

    # Create YOUR role (needed to connect and run GRANT statements)
    if current_user in existing_roles:
        print(f"  Postgres role for '{current_user}' already exists (OK)")
    else:
        try:
            w.database.create_database_instance_role(
                instance_name=LAKEBASE_NAME,
                database_instance_role=DatabaseInstanceRole(
                    name=current_user,
                    identity_type=DatabaseInstanceRoleIdentityType.USER,
                ),
            )
            print(f"  Created Postgres role for user '{current_user}'")
        except Exception as e:
            if "already exists" in str(e).lower():
                print(f"  Postgres role for '{current_user}' already exists (OK)")
            else:
                print(f"  Warning creating user role: {e}")

    # Create SP role (needed for the agent to connect at runtime)
    if sp_role in existing_roles:
        print(f"  Postgres role for SP '{sp_role}' already exists (OK)")
    else:
        try:
            w.database.create_database_instance_role(
                instance_name=LAKEBASE_NAME,
                database_instance_role=DatabaseInstanceRole(
                    name=sp_role,
                    identity_type=DatabaseInstanceRoleIdentityType.SERVICE_PRINCIPAL,
                ),
            )
            print(f"  Created Postgres role for SP '{sp_role}'")
        except Exception as e:
            if "already exists" in str(e).lower():
                print(f"  Postgres role for SP '{sp_role}' already exists (OK)")
            else:
                print(f"  Warning creating SP role: {e}")

    # ---- 2. Grant database permissions via Postgres connection ----
    # Now that both roles exist, connect as the current user and GRANT permissions to the SP.
    # Permission: ALL PRIVILEGES on public schema (SELECT, INSERT, UPDATE, DELETE, CREATE)
    # Scope: public schema only — does not affect other schemas
    # Why: LangGraph PostgresSaver needs CREATE TABLE + full CRUD on checkpoint tables
    # To reduce: replace ALL PRIVILEGES with specific grants on checkpoint_* tables only
    print(f"\n--- Step 2: Grant database permissions (Postgres connection) ---")
    print(f"  Connecting to Lakebase as '{current_user}'...")

    cred = w.database.generate_database_credential(
        request_id=str(_uuid.uuid4()),
        instance_names=[LAKEBASE_NAME],
    )

    conn = psycopg.connect(
        f"dbname=databricks_postgres user={current_user} host={LAKEBASE_HOST} sslmode=require",
        password=cred.token,
    )
    conn.autocommit = True

    with conn.cursor() as cur:
        grant_statements = [
            f'GRANT USAGE ON SCHEMA public TO "{sp_role}"',
            f'GRANT CREATE ON SCHEMA public TO "{sp_role}"',
            f'GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA public TO "{sp_role}"',
            f'GRANT ALL PRIVILEGES ON ALL SEQUENCES IN SCHEMA public TO "{sp_role}"',
            f'ALTER DEFAULT PRIVILEGES IN SCHEMA public GRANT ALL PRIVILEGES ON TABLES TO "{sp_role}"',
            f'ALTER DEFAULT PRIVILEGES IN SCHEMA public GRANT ALL PRIVILEGES ON SEQUENCES TO "{sp_role}"',
        ]

        print(f"  Granting database permissions to SP...")
        for stmt in grant_statements:
            try:
                cur.execute(stmt)
                print(f"    OK: {stmt[:70]}...")
            except Exception as e:
                print(f"    Note: {e}")

    conn.close()

    # ---- 3. (Optional) Grant project-level permissions to SP via REST API ----
    # This grants CAN_MANAGE on the Lakebase *instance* so the SP can call
    # generate_database_credential() at runtime.
    #
    # NOTE: This API call does not work reliably on all Lakebase Provisioned
    # instances. If it fails, the SP will still work AS LONG AS you have granted
    # it access via the Lakebase App UI (Settings > Permissions > Add SP).
    # Steps 1 and 2 above are what actually matter for setup.
    print(f"\n--- Step 3: Project-level permissions (REST API, optional) ---")
    try:
        _instance = w.database.get_database_instance(name=LAKEBASE_NAME)
        _instance_uid = _instance.uid

        w.api_client.do(
            "PUT",
            f"/api/2.0/permissions/database-instances/{_instance_uid}",
            body={
                "access_control_list": [
                    {
                        "service_principal_name": sp_role,
                        "permission_level": "CAN_MANAGE"
                    }
                ]
            }
        )
        print(f"  Granted CAN_MANAGE on Lakebase instance '{LAKEBASE_NAME}'")
    except Exception as e:
        error_str = str(e)
        if "already has" in error_str.lower() or "PERMISSION_ALREADY_EXISTS" in error_str:
            print(f"  SP already has CAN_MANAGE on Lakebase instance (OK)")
        else:
            print(f"  Could not grant via API (common for Provisioned instances).")
            print(f"  ACTION REQUIRED: Grant SP access manually in the Lakebase App UI:")
            print(f"    1. Open the Lakebase App > click '{LAKEBASE_NAME}'")
            print(f"    2. Go to Settings > Permissions")
            print(f"    3. Add service principal '{SERVICE_PRINCIPAL_NAME}' with CAN_USE or CAN_MANAGE")
            print(f"  Without this, the deployed agent may fail to generate database credentials.")

    print(f"\n{'='*70}")
    print(f" LAKEBASE SETUP COMPLETE")
    print(f"{'='*70}")
    print(f"  Instance:    {LAKEBASE_NAME} ({LAKEBASE_INSTANCE_ID})")
    print(f"  Host:        {LAKEBASE_HOST}")
    print(f"  Your role:   {current_user} (created via SDK)")
    print(f"  SP role:     {sp_role} (created via SDK)")
    print(f"  Database:    ALL PRIVILEGES on public schema (granted via SQL)")
    print(f"  Project:     See Step 3 above — grant via UI if API failed")

In [ ]:
# Create alignment tracking table for scheduled job filtering
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {ALIGNMENT_RUNS_TABLE} (
        run_id STRING COMMENT 'Unique identifier for the alignment run',
        alignment_timestamp TIMESTAMP COMMENT 'When the alignment job completed',
        traces_processed INT COMMENT 'Number of traces processed in this run',
        status STRING COMMENT 'Status of the run: SUCCESS, FAILED, SKIPPED'
    )
    COMMENT 'Tracks alignment job runs for incremental processing'
""")

print(f"Alignment tracking table ready: {ALIGNMENT_RUNS_TABLE}")

In [ ]:
# Save configuration to JSON file
config_dir = Path("config")
config_dir.mkdir(exist_ok=True)

config_file = config_dir / "atbat_assistant.json"
with open(config_file, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Configuration saved to: {config_file.absolute()}")
print(f"\nTo load in other notebooks, use:")
print(f"  import json")
print(f"  from pathlib import Path")
print(f"  CONFIG = json.loads(Path('config/atbat_assistant.json').read_text())")

In [ ]:
print("Setup complete. All notebooks load config from config/atbat_assistant.json.")